In [ ]:   # This Python 3 environment comes with many helpful analytics libraries   installed # It is defined by the kaggle/python Docker image:   https://github.com/kaggle/docker-python # For example, here's several helpful packages to load import numpy as np # linear algebra import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv) # Input data files are available in the read-only "../input/" directory # For example, running this (by clicking run or pressing Shift+Enter)   will list all files under the input directory import os for dirname, _, filenames in os.walk('/kaggle/input'):      for filename in filenames:          print(os.path.join(dirname, filename)) # You can write up to 20GB to the current directory (/kaggle/working/)   that gets preserved as output when you create a version using "Save &   Run All"  # You can also write temporary files to /kaggle/temp/, but they won't be   saved outside of the current session # Use the kagglehub client library to attach Kaggle resources like   competitions, datasets, and models to your session # Learn more about kagglehub:   https://github.com/Kaggle/kagglehub/blob/main/README.md import kagglehub # kagglehub.dataset_download('<owner>/<dataset-slug>')  In [4]:   import os ALIGNED_ROOT = "/kaggle/input/datasets/srujanamannepuli/taligned" REAL_DIR = os.path.join(ALIGNED_ROOT, "Real") FAKE_DIR = os.path.join(ALIGNED_ROOT, "Fake") print("Real:", os.path.exists(REAL_DIR)) print("Fake:", os.path.exists(FAKE_DIR))  Real: True Fake: True  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   1 / 77

--- End of Page 1 ---

In [5]:   real_video = sorted(os.listdir(REAL_DIR))[0] real_video_dir = os.path.join(      REAL_DIR,      real_video ) frames = sorted([      f for f in os.listdir(real_video_dir)      if f.endswith(".jpg") ]) print("Video:", real_video) print("Frames:", frames) print("Number of frames:", len(frames))  Video: 00000 Frames: ['frame_0048.jpg', 'frame_0065.jpg', 'frame_0066.jpg',   'frame_0242.jpg', 'frame_0302.jpg'] Number of frames: 5  In [6]:   import os import cv2 import torch import timm import numpy as np from PIL import Image from tqdm import tqdm device = torch.device(      "cuda" if torch.cuda.is_available() else "cpu" ) print("Device:", device) temporal_efficientnet = timm.create_model(      "efficientnet_b4",      pretrained=True,      num_classes=0 ).to(device).eval() print("EfficientNet-B4 ready")  Device: cuda Warning: You are sending unauthenticated requests to the HF   Hub. Please set a HF_TOKEN to enable higher rate limits and   faster downloads. model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s] EfficientNet-B4 ready  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   2 / 77

--- End of Page 2 ---

In [11]:   from torchvision import transforms gray_transform = transforms.Compose([      transforms.Grayscale(num_output_channels=3),      transforms.Resize((224, 224)),      transforms.ToTensor(),      transforms.Normalize(          mean=[0.485, 0.456, 0.406],          std=[0.229, 0.224, 0.225]      ) ])  In [12]:   sample_video = sorted(os.listdir(REAL_DIR))[0] sample_dir = os.path.join(      REAL_DIR,      sample_video ) frame_files = sorted([      f for f in os.listdir(sample_dir)      if f.lower().endswith(".jpg") ]) print("Video:", sample_video) print("Frames:", frame_files) print("Number of frames:", len(frame_files))  Video: 00000 Frames: ['frame_0048.jpg', 'frame_0065.jpg', 'frame_0066.jpg',   'frame_0242.jpg', 'frame_0302.jpg'] Number of frames: 5  In [13]:   gray_frames = [] for frame_file in frame_files:      path = os.path.join(          sample_dir,          frame_file      )      image = Image.open(path).convert("RGB")      gray_image = image.convert("L")      gray_frames.append(gray_image) print("Loaded grayscale frames:", len(gray_frames)) print("Frame size:", gray_frames[0].size)  Loaded grayscale frames: 5 Frame size: (224, 224)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   3 / 77

--- End of Page 3 ---

In [14]:   frame_differences = [] for i in range(1, len(gray_frames)):      current = np.array(          gray_frames[i]      ).astype(np.float32)      previous = np.array(          gray_frames[i - 1]      ).astype(np.float32)      difference = np.abs(          current - previous      )      difference = np.clip(          difference,          0,          255      ).astype(np.uint8)      frame_differences.append(          Image.fromarray(difference)      ) print(      "Frame differences:",      len(frame_differences) ) print(      "Difference size:",      frame_differences[0].size )  Frame differences: 4 Difference size: (224, 224)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   4 / 77

--- End of Page 4 ---

In [15]:   temporal_features = [] with torch.no_grad():      for diff_image in frame_differences:          tensor = gray_transform(              diff_image          ).unsqueeze(0).to(device)          features = temporal_efficientnet(              tensor          )          temporal_features.append(              features.squeeze(0).cpu()          ) temporal_features = torch.stack(      temporal_features ) print(      "Temporal CNN features:",      temporal_features.shape )  Temporal CNN features: torch.Size([4, 1792])  In [16]:   import torch.nn as nn temporal_bilstm = nn.LSTM(      input_size=1792,      hidden_size=256,      num_layers=2,      batch_first=True,      bidirectional=True ).to(device) print("BiLSTM initialized.")  BiLSTM initialized.  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   5 / 77

--- End of Page 5 ---

In [17]:   with torch.no_grad():      lstm_input = temporal_features.unsqueeze(0).to(device)      lstm_output, _ = temporal_bilstm(          lstm_input      ) temporal_embedding = lstm_output.squeeze(0).cpu() print(      "Temporal embedding:",      temporal_embedding.shape )  Temporal embedding: torch.Size([4, 512])  In [18]:   TEMPORAL_ROOT = "/kaggle/working/temporal_features" os.makedirs(TEMPORAL_ROOT, exist_ok=True) TEMPORAL_CHECKPOINT = "/kaggle/working/temporal_checkpoint.csv" if os.path.exists(TEMPORAL_CHECKPOINT):      temporal_ckpt = pd.read_csv(TEMPORAL_CHECKPOINT)      processed_temporal = set(          temporal_ckpt["video_id"].astype(str)      )      temporal_results = temporal_ckpt.to_dict("records")      print(          "Resuming:",          len(processed_temporal),          "videos"      ) else:      processed_temporal = set()      temporal_results = []      print("Starting temporal extraction...")  Starting temporal extraction...  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   6 / 77

--- End of Page 6 ---

In [19]:   SAVE_EVERY = 100 temporal_counter = 0 for label, label_dir in [      ("Real", REAL_DIR),      ("Fake", FAKE_DIR) ]:      video_ids = sorted(os.listdir(label_dir))      for video_id in tqdm(          video_ids,          desc=f"Temporal {label}"      ):          if video_id in processed_temporal:              continue          video_dir = os.path.join(              label_dir,              video_id          )          if not os.path.isdir(video_dir):              continue          frame_files = sorted([              f for f in os.listdir(video_dir)              if f.lower().endswith(".jpg")          ])          # We require exactly 5 frames          if len(frame_files) != 5:              temporal_results.append({                  "video_id": video_id,                  "label": label,                  "num_frames": len(frame_files),                  "status": "skipped_not_5_frames"              })              processed_temporal.add(video_id)              continue          try:              # -----------------------------------------              # Load 5 grayscale frames              # -----------------------------------------              gray_frames = []              for frame_file in frame_files:                  path = os.path.join(                      video_dir,                      frame_file  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   7 / 77

--- End of Page 7 ---

)                  image = Image.open(                      path                  ).convert("RGB")                  gray_frames.append(                      image.convert("L")                  )              # -----------------------------------------              # 5 frames → 4 differences              # -----------------------------------------              frame_differences = []              for i in range(1, 5):                  current = np.asarray(                      gray_frames[i]                  ).astype(np.float32)                  previous = np.asarray(                      gray_frames[i - 1]                  ).astype(np.float32)                  difference = np.abs(                      current - previous                  )                  difference = np.clip(                      difference,                      0,                      255                  ).astype(np.uint8)                  frame_differences.append(                      Image.fromarray(                          difference                      )                  )              # -----------------------------------------              # EfficientNet-B4              # -----------------------------------------              cnn_features = []              with torch.no_grad():                  for diff_image in frame_differences:                      tensor = gray_transform(                          diff_image                      ).unsqueeze(0).to(device)                      feat = temporal_efficientnet(  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   8 / 77

--- End of Page 8 ---

tensor                      )                      cnn_features.append(                          feat.squeeze(0)                      )              cnn_features = torch.stack(                  cnn_features              ).to(device)              # Shape: 4 × 1792              # -----------------------------------------              # BiLSTM              # -----------------------------------------              with torch.no_grad():                  lstm_input = cnn_features.unsqueeze(0)                  lstm_output, _ = temporal_bilstm(                      lstm_input                  )              temporal_embedding = (                  lstm_output                  .squeeze(0)                  .cpu()                  .numpy()              )              # Shape: 4 × 512              # -----------------------------------------              # Save              # -----------------------------------------              feature_path = os.path.join(                  TEMPORAL_ROOT,                  f"{label}_{video_id}.npy"              )              np.save(                  feature_path,                  temporal_embedding              )              temporal_results.append({                  "video_id": video_id,                  "label": label,                  "num_frames": 5,                  "num_differences": 4,                  "feature_shape": "4x512",                  "feature_path": feature_path,                  "status": "processed"  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   9 / 77

--- End of Page 9 ---

})              processed_temporal.add(                  video_id              )              temporal_counter += 1              # -----------------------------------------              # Checkpoint              # -----------------------------------------              if temporal_counter % SAVE_EVERY == 0:                  pd.DataFrame(                      temporal_results                  ).to_csv(                      TEMPORAL_CHECKPOINT,                      index=False                  )                  print(                      f"\nCheckpoint: "                      f"{len(processed_temporal)} videos"                  )          except Exception as e:              temporal_results.append({                  "video_id": video_id,                  "label": label,                  "status": f"error: {str(e)}"              })              processed_temporal.add(                  video_id              ) # Final checkpoint pd.DataFrame(      temporal_results ).to_csv(      TEMPORAL_CHECKPOINT,      index=False ) print("\n================================") print("TEMPORAL EXTRACTION COMPLETED") print("================================") print(      "Videos processed:",      len(processed_temporal)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   10 / 77

--- End of Page 10 ---

) print(      "Output:",      TEMPORAL_ROOT )  Temporal Real: 100%|██████████| 888/888 [01:16<00:00,   11.55it/s] Temporal Fake: 100%|██████████| 5639/5639 [08:18<00:00,   11.32it/s] --------------------------------------------------------------- ------------ NameError                                 Traceback (most   recent call last) /tmp/ipykernel_58/4267185338.py in <cell line: 0>()      214 # Final checkpoint      215  --> 216 pd.DataFrame(      217     temporal_results      218 ).to_csv( NameError: name 'pd' is not defined  In [21]:   import pandas as pd pd.DataFrame(      temporal_results ).to_csv(      TEMPORAL_CHECKPOINT,      index=False ) print("Final checkpoint saved successfully.") print("Videos processed:", len(processed_temporal))  Final checkpoint saved successfully. Videos processed: 6527  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   11 / 77

--- End of Page 11 ---

In [22]:   temporal_files = [      f for f in os.listdir(TEMPORAL_ROOT)      if f.endswith(".npy") ] test_temporal = np.load(      os.path.join(          TEMPORAL_ROOT,          temporal_files[0]      ) ) print("Temporal feature shape:", test_temporal.shape)  Temporal feature shape: (4, 512)  In [23]:   TEMPORAL_CNN_ROOT = "/kaggle/working/temporal_cnn_features" os.makedirs(TEMPORAL_CNN_ROOT, exist_ok=True) TEMPORAL_CNN_CHECKPOINT = (      "/kaggle/working/temporal_cnn_checkpoint.csv" ) if os.path.exists(TEMPORAL_CNN_CHECKPOINT):      cnn_ckpt = pd.read_csv(          TEMPORAL_CNN_CHECKPOINT      )      processed_cnn = set(          cnn_ckpt["video_id"].astype(str)      )      cnn_results = cnn_ckpt.to_dict(          "records"      )      print(          "Resuming:",          len(processed_cnn),          "videos"      ) else:      processed_cnn = set()      cnn_results = []      print("Starting Temporal CNN extraction...")  Starting Temporal CNN extraction...  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   12 / 77

--- End of Page 12 ---

In [24]:   SAVE_EVERY = 100 cnn_counter = 0 for label, label_dir in [      ("Real", REAL_DIR),      ("Fake", FAKE_DIR) ]:      video_ids = sorted(          os.listdir(label_dir)      )      for video_id in tqdm(          video_ids,          desc=f"Temporal CNN {label}"      ):          if video_id in processed_cnn:              continue          video_dir = os.path.join(              label_dir,              video_id          )          if not os.path.isdir(video_dir):              continue          frame_files = sorted([              f for f in os.listdir(video_dir)              if f.lower().endswith(".jpg")          ])          if len(frame_files) != 5:              cnn_results.append({                  "video_id": video_id,                  "label": label,                  "num_frames": len(frame_files),                  "status": "skipped_not_5_frames"              })              processed_cnn.add(video_id)              continue          try:              # -------------------------------              # Load 5 grayscale frames              # -------------------------------              gray_frames = []              for frame_file in frame_files:                  path = os.path.join(                      video_dir,  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   13 / 77

--- End of Page 13 ---

frame_file                  )                  image = Image.open(                      path                  ).convert("RGB")                  gray_frames.append(                      image.convert("L")                  )              # -------------------------------              # 5 frames → 4 differences              # -------------------------------              frame_differences = []              for i in range(1, 5):                  current = np.asarray(                      gray_frames[i]                  ).astype(np.float32)                  previous = np.asarray(                      gray_frames[i - 1]                  ).astype(np.float32)                  difference = np.abs(                      current - previous                  )                  difference = np.clip(                      difference,                      0,                      255                  ).astype(np.uint8)                  frame_differences.append(                      Image.fromarray(                          difference                      )                  )              # -------------------------------              # EfficientNet-B4              # -------------------------------              cnn_features = []              with torch.no_grad():                  for diff_image in frame_differences:                      tensor = gray_transform(                          diff_image                      ).unsqueeze(0).to(device)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   14 / 77

--- End of Page 14 ---

features = temporal_efficientnet(                          tensor                      )                      cnn_features.append(                          features.squeeze(0)                          .cpu()                          .numpy()                      )              cnn_features = np.stack(                  cnn_features              )              # Shape = (4, 1792)              feature_path = os.path.join(                  TEMPORAL_CNN_ROOT,                  f"{label}_{video_id}.npy"              )              np.save(                  feature_path,                  cnn_features              )              cnn_results.append({                  "video_id": video_id,                  "label": label,                  "num_frames": 5,                  "num_differences": 4,                  "feature_shape": "4x1792",                  "feature_path": feature_path,                  "status": "processed"              })              processed_cnn.add(                  video_id              )              cnn_counter += 1              # -------------------------------              # Checkpoint              # -------------------------------              if cnn_counter % SAVE_EVERY == 0:                  pd.DataFrame(                      cnn_results                  ).to_csv(                      TEMPORAL_CNN_CHECKPOINT,                      index=False                  )                  print(                      f"\nCheckpoint saved: "  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   15 / 77

--- End of Page 15 ---

f"{len(processed_cnn)} videos"                  )          except Exception as e:              cnn_results.append({                  "video_id": video_id,                  "label": label,                  "status": f"error: {str(e)}"              })              processed_cnn.add(                  video_id              ) # Final checkpoint pd.DataFrame(      cnn_results ).to_csv(      TEMPORAL_CNN_CHECKPOINT,      index=False ) print("\n====================================") print("TEMPORAL CNN EXTRACTION COMPLETED") print("====================================") print(      "Videos processed:",      len(processed_cnn) ) print(      "Output:",      TEMPORAL_CNN_ROOT )  Temporal CNN Real:  11%|█ ▏         | 102/888 [00:07<01:00,   12.93it/s] Checkpoint saved: 100 videos Temporal CNN Real:  23%|██ ▎        | 204/888 [00:15<00:52,   13.07it/s] Checkpoint saved: 202 videos Temporal CNN Real:  34%|███ ▍       | 304/888 [00:23<00:45,   12.84it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   16 / 77

--- End of Page 16 ---

Checkpoint saved: 302 videos Temporal CNN Real:  46%|████▌     | 410/888 [00:31<00:37,   12.84it/s] Checkpoint saved: 408 videos Temporal CNN Real:  57%|█████ ▋     | 510/888 [00:38<00:29,   12.71it/s] Checkpoint saved: 508 videos Temporal CNN Real:  69%|██████ ▊    | 610/888 [00:46<00:23,   12.07it/s] Checkpoint saved: 608 videos Temporal CNN Real:  80%|███████ ▉   | 710/888 [00:54<00:14,   12.00it/s] Checkpoint saved: 708 videos Temporal CNN Real:  91%|█████████ | 810/888 [01:02<00:06,   12.79it/s] Checkpoint saved: 808 videos Temporal CNN Real: 100%|██████████| 888/888 [01:08<00:00,   12.99it/s] Temporal CNN Fake:   0%|          | 22/5639 [00:01<07:29,   12.51it/s] Checkpoint saved: 909 videos Temporal CNN Fake:   2%| ▏          | 122/5639 [00:09<07:16,   12.63it/s] Checkpoint saved: 1009 videos Temporal CNN Fake:   4%| ▍          | 222/5639 [00:17<07:20,    8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   17 / 77

--- End of Page 17 ---

12.30it/s] Checkpoint saved: 1109 videos Temporal CNN Fake:   6%|▌         | 322/5639 [00:25<07:03,   12.55it/s] Checkpoint saved: 1209 videos Temporal CNN Fake:   7%| ▋          | 422/5639 [00:33<06:57,   12.50it/s] Checkpoint saved: 1309 videos Temporal CNN Fake:   9%| ▉          | 522/5639 [00:41<06:50,   12.47it/s] Checkpoint saved: 1409 videos Temporal CNN Fake:  11%|█         | 622/5639 [00:49<06:43,   12.44it/s] Checkpoint saved: 1509 videos Temporal CNN Fake:  13%|█ ▎         | 722/5639 [00:56<06:21,   12.90it/s] Checkpoint saved: 1609 videos Temporal CNN Fake:  15%|█ ▍         | 822/5639 [01:04<06:21,   12.63it/s] Checkpoint saved: 1709 videos Temporal CNN Fake:  16%|█ ▋         | 922/5639 [01:12<06:05,   12.90it/s] Checkpoint saved: 1809 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   18 / 77

--- End of Page 18 ---

Temporal CNN Fake:  18%|█ ▊         | 1022/5639 [01:20<06:08,   12.54it/s] Checkpoint saved: 1909 videos Temporal CNN Fake:  20%|█ ▉         | 1122/5639 [01:28<06:28,   11.61it/s] Checkpoint saved: 2009 videos Temporal CNN Fake:  22%|██ ▏        | 1222/5639 [01:36<05:50,   12.62it/s] Checkpoint saved: 2109 videos Temporal CNN Fake:  23%|██ ▎        | 1322/5639 [01:44<05:53,   12.20it/s] Checkpoint saved: 2209 videos Temporal CNN Fake:  25%|██▌       | 1422/5639 [01:52<05:37,   12.50it/s] Checkpoint saved: 2309 videos Temporal CNN Fake:  27%|██ ▋        | 1522/5639 [02:00<05:24,   12.70it/s] Checkpoint saved: 2409 videos Temporal CNN Fake:  29%|██ ▉        | 1622/5639 [02:07<05:30,   12.15it/s] Checkpoint saved: 2509 videos Temporal CNN Fake:  31%|███       | 1722/5639 [02:15<05:10,   12.62it/s] Checkpoint saved: 2609 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   19 / 77

--- End of Page 19 ---

Temporal CNN Fake:  32%|███ ▏       | 1822/5639 [02:23<05:06,   12.44it/s] Checkpoint saved: 2709 videos Temporal CNN Fake:  34%|███ ▍       | 1922/5639 [02:31<04:55,   12.59it/s] Checkpoint saved: 2809 videos Temporal CNN Fake:  36%|███▌      | 2022/5639 [02:39<04:59,   12.07it/s] Checkpoint saved: 2909 videos Temporal CNN Fake:  38%|███ ▊       | 2122/5639 [02:47<04:37,   12.66it/s] Checkpoint saved: 3009 videos Temporal CNN Fake:  39%|███ ▉       | 2222/5639 [02:54<04:29,   12.66it/s] Checkpoint saved: 3109 videos Temporal CNN Fake:  41%|████      | 2322/5639 [03:02<04:21,   12.67it/s] Checkpoint saved: 3209 videos Temporal CNN Fake:  43%|████ ▎      | 2422/5639 [03:10<04:11,   12.77it/s] Checkpoint saved: 3309 videos Temporal CNN Fake:  45%|████ ▍      | 2522/5639 [03:18<04:18,   12.08it/s] Checkpoint saved: 3409 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   20 / 77

--- End of Page 20 ---

Temporal CNN Fake:  46%|████ ▋      | 2622/5639 [03:25<04:01,   12.47it/s] Checkpoint saved: 3509 videos Temporal CNN Fake:  48%|████ ▊      | 2722/5639 [03:33<03:58,   12.25it/s] Checkpoint saved: 3609 videos Temporal CNN Fake:  50%|█████     | 2822/5639 [03:41<03:49,   12.29it/s] Checkpoint saved: 3709 videos Temporal CNN Fake:  52%|█████ ▏     | 2922/5639 [03:49<03:37,   12.49it/s] Checkpoint saved: 3809 videos Temporal CNN Fake:  54%|█████ ▎     | 3022/5639 [03:57<03:33,   12.25it/s] Checkpoint saved: 3909 videos Temporal CNN Fake:  55%|█████▌    | 3122/5639 [04:05<03:25,   12.26it/s] Checkpoint saved: 4009 videos Temporal CNN Fake:  57%|█████ ▋     | 3222/5639 [04:13<03:19,   12.14it/s] Checkpoint saved: 4109 videos Temporal CNN Fake:  59%|█████ ▉     | 3322/5639 [04:21<03:13,   11.98it/s] Checkpoint saved: 4209 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   21 / 77

--- End of Page 21 ---

Temporal CNN Fake:  61%|██████    | 3422/5639 [04:29<03:04,   12.01it/s] Checkpoint saved: 4309 videos Temporal CNN Fake:  62%|██████ ▏    | 3524/5639 [04:36<02:51,   12.33it/s] Checkpoint saved: 4411 videos Temporal CNN Fake:  64%|██████ ▍    | 3625/5639 [04:44<02:39,   12.62it/s] Checkpoint saved: 4512 videos Temporal CNN Fake:  66%|██████▌   | 3725/5639 [04:52<02:35,   12.27it/s] Checkpoint saved: 4612 videos Temporal CNN Fake:  68%|██████ ▊    | 3825/5639 [05:00<02:30,   12.04it/s] Checkpoint saved: 4712 videos Temporal CNN Fake:  70%|██████ ▉    | 3925/5639 [05:08<02:28,   11.54it/s] Checkpoint saved: 4812 videos Temporal CNN Fake:  71%|███████ ▏   | 4025/5639 [05:16<02:11,   12.31it/s] Checkpoint saved: 4912 videos Temporal CNN Fake:  73%|███████ ▎   | 4125/5639 [05:24<02:04,   12.19it/s] Checkpoint saved: 5012 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   22 / 77

--- End of Page 22 ---

Temporal CNN Fake:  75%|███████ ▍   | 4225/5639 [05:31<01:53,   12.46it/s] Checkpoint saved: 5112 videos Temporal CNN Fake:  77%|███████ ▋   | 4325/5639 [05:39<01:50,   11.84it/s] Checkpoint saved: 5212 videos Temporal CNN Fake:  78%|███████ ▊   | 4425/5639 [05:47<01:40,   12.04it/s] Checkpoint saved: 5312 videos Temporal CNN Fake:  80%|████████  | 4525/5639 [05:55<01:32,   12.06it/s] Checkpoint saved: 5412 videos Temporal CNN Fake:  82%|████████ ▏  | 4625/5639 [06:03<01:23,   12.15it/s] Checkpoint saved: 5512 videos Temporal CNN Fake:  84%|████████ ▍  | 4725/5639 [06:11<01:14,   12.24it/s] Checkpoint saved: 5612 videos Temporal CNN Fake:  86%|████████▌ | 4825/5639 [06:19<01:14,   10.99it/s] Checkpoint saved: 5712 videos Temporal CNN Fake:  87%|████████ ▋  | 4925/5639 [06:26<00:58,   12.28it/s] Checkpoint saved: 5812 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   23 / 77

--- End of Page 23 ---

Temporal CNN Fake:  89%|████████ ▉  | 5027/5639 [06:34<00:50,   12.14it/s] Checkpoint saved: 5914 videos Temporal CNN Fake:  91%|█████████ | 5127/5639 [06:42<00:44,   11.44it/s] Checkpoint saved: 6014 videos Temporal CNN Fake:  93%|█████████ ▎ | 5227/5639 [06:50<00:34,   12.05it/s] Checkpoint saved: 6114 videos Temporal CNN Fake:  94%|█████████ ▍ | 5327/5639 [06:58<00:26,   11.62it/s] Checkpoint saved: 6214 videos Temporal CNN Fake:  96%|█████████▌| 5427/5639 [07:06<00:17,   12.03it/s] Checkpoint saved: 6314 videos Temporal CNN Fake:  98%|█████████ ▊ | 5527/5639 [07:14<00:09,   12.11it/s] Checkpoint saved: 6414 videos Temporal CNN Fake: 100%|█████████ ▉ | 5627/5639 [07:21<00:00,   12.23it/s] Checkpoint saved: 6514 videos Temporal CNN Fake: 100%|██████████| 5639/5639 [07:22<00:00,   12.74it/s] ==================================== TEMPORAL CNN EXTRACTION COMPLETED  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   24 / 77

--- End of Page 24 ---

==================================== Videos processed: 6527 Output: /kaggle/working/temporal_cnn_features  In [25]:   import torch import torch.nn as nn class TemporalBiLSTM(nn.Module):      def __init__(          self,          input_size=1792,          hidden_size=256,          num_layers=2      ):          super().__init__()          self.bilstm = nn.LSTM(              input_size=input_size,              hidden_size=hidden_size,              num_layers=num_layers,              batch_first=True,              bidirectional=True          )      def forward(self, x):          # x: (batch, 4, 1792)          output, _ = self.bilstm(x)          # output: (batch, 4, 512)          return output temporal_bilstm_model = TemporalBiLSTM(      input_size=1792,      hidden_size=256,      num_layers=2 ).to(device) print(temporal_bilstm_model)  TemporalBiLSTM(    (bilstm): LSTM(1792, 256, num_layers=2, batch_first=True,   bidirectional=True) )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   25 / 77

--- End of Page 25 ---

In [27]:   import os import numpy as np temporal_cnn_files = [      f for f in os.listdir(TEMPORAL_CNN_ROOT)      if f.endswith(".npy") ] print("Number of feature files:", len(temporal_cnn_files)) test_file = temporal_cnn_files[0] test_cnn = np.load(      os.path.join(          TEMPORAL_CNN_ROOT,          test_file      ) ) print("Test file:", test_file) print("Temporal CNN feature shape:", test_cnn.shape)  Number of feature files: 6513 Test file: Fake_id4_id20_0006.npy Temporal CNN feature shape: (4, 1792)  In [28]:   test_feature = torch.tensor(      test_cnn,      dtype=torch.float32 ).unsqueeze(0).to(device) with torch.no_grad():      temporal_output = temporal_bilstm_model(          test_feature      ) print("Input:", test_feature.shape) print("BiLSTM output:", temporal_output.shape)  Input: torch.Size([1, 4, 1792]) BiLSTM output: torch.Size([1, 4, 512])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   26 / 77

--- End of Page 26 ---

In [29]:   test_feature = torch.tensor(      test_cnn,      dtype=torch.float32 ).unsqueeze(0).to(device) with torch.no_grad():      temporal_output = temporal_bilstm_model(          test_feature      ) print("Input:", test_feature.shape) print("BiLSTM output:", temporal_output.shape)  Input: torch.Size([1, 4, 1792]) BiLSTM output: torch.Size([1, 4, 512])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   27 / 77

--- End of Page 27 ---

In [30]:   import torch import torch.nn as nn import torch.nn.functional as F class AdaptiveCrossAttentionFusion(nn.Module):      def __init__(          self,          feature_dim=512,          num_heads=8      ):          super().__init__()          # Spatial → Q          self.q_proj = nn.Linear(              feature_dim,              feature_dim          )          # Temporal → K          self.k_proj = nn.Linear(              feature_dim,              feature_dim          )          # Temporal → V          self.v_proj = nn.Linear(              feature_dim,              feature_dim          )          self.cross_attention = nn.MultiheadAttention(              embed_dim=feature_dim,              num_heads=num_heads,              batch_first=True          )          self.norm = nn.LayerNorm(              feature_dim          )          # Adaptive gating          self.gate = nn.Sequential(              nn.Linear(                  feature_dim * 2,                  feature_dim              ),              nn.Sigmoid()          )      def forward(          self,          spatial,          temporal      ):  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   28 / 77

--- End of Page 28 ---

# spatial  = (B, 5, 512)          # temporal = (B, 4, 512)          Q = self.q_proj(spatial)          K = self.k_proj(temporal)          V = self.v_proj(temporal)          attention_output, attention_weights = (              self.cross_attention(                  Q,                  K,                  V              )          )          # Residual + normalization          attention_output = self.norm(              spatial + attention_output          )          # Adaptive gate          pooled_spatial = spatial.mean(              dim=1          )          pooled_attention = attention_output.mean(              dim=1          )          gate_input = torch.cat(              [                  pooled_spatial,                  pooled_attention              ],              dim=1          )          gate = self.gate(              gate_input          )          fused = (              gate * pooled_attention              +              (1 - gate) * pooled_spatial          )          return fused, attention_weights  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   29 / 77

--- End of Page 29 ---

In [31]:   fusion_model = AdaptiveCrossAttentionFusion(      feature_dim=512,      num_heads=8 ).to(device) # Dummy inputs only for architecture verification spatial_test = torch.randn(      1, 5, 512 ).to(device) temporal_test = torch.randn(      1, 4, 512 ).to(device) with torch.no_grad():      fused_test, attention_weights = fusion_model(          spatial_test,          temporal_test      ) print("Spatial:", spatial_test.shape) print("Temporal:", temporal_test.shape) print("Fused:", fused_test.shape) print("Attention:", attention_weights.shape)  Spatial: torch.Size([1, 5, 512]) Temporal: torch.Size([1, 4, 512]) Fused: torch.Size([1, 512]) Attention: torch.Size([1, 5, 4])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   30 / 77

--- End of Page 30 ---

In [32]:   class ClassificationHead(nn.Module):      def __init__(self):          super().__init__()          self.fc1 = nn.Linear(              512,              256          )          self.dropout = nn.Dropout(              0.5          )          self.fc2 = nn.Linear(              256,              128          )          self.fc3 = nn.Linear(              128,              2          )      def forward(self, x):          x = F.relu(              self.fc1(x)          )          x = self.dropout(x)          x = F.relu(              self.fc2(x)          )          logits = self.fc3(x)          probabilities = F.softmax(              logits,              dim=1          )          return logits, probabilities  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   31 / 77

--- End of Page 31 ---

In [33]:   classifier = ClassificationHead().to(device) with torch.no_grad():      logits, probabilities = classifier(          fused_test      ) print("Fused feature:", fused_test.shape) print("Logits:", logits.shape) print("Probabilities:", probabilities.shape) print("\nProbabilities:") print(probabilities) print(      "\nProbability sum:",      probabilities.sum(dim=1) )  Fused feature: torch.Size([1, 512]) Logits: torch.Size([1, 2]) Probabilities: torch.Size([1, 2]) Probabilities: tensor([[0.5071, 0.4929]], device='cuda:0') Probability sum: tensor([1.], device='cuda:0')  In [36]:   import os import torch import numpy as np from PIL import Image from tqdm import tqdm TEMPORAL_FAST_ROOT = "/kaggle/working/temporal_cnn_features_fast" os.makedirs(TEMPORAL_FAST_ROOT, exist_ok=True) print("Output:", TEMPORAL_FAST_ROOT) print("GPU:", torch.cuda.get_device_name(0))  Output: /kaggle/working/temporal_cnn_features_fast GPU: Tesla T4  In [37]:   temporal_efficientnet.eval() print("EfficientNet-B4 ready")  EfficientNet-B4 ready  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   32 / 77

--- End of Page 32 ---

In [38]:   sample_video_id = sorted(os.listdir(REAL_DIR))[0] sample_video_dir = os.path.join(      REAL_DIR,      sample_video_id ) frame_files = sorted([      f for f in os.listdir(sample_video_dir)      if f.lower().endswith(".jpg") ]) print("Video:", sample_video_id) print("Frames:", frame_files)  Video: 00000 Frames: ['frame_0048.jpg', 'frame_0065.jpg', 'frame_0066.jpg',   'frame_0242.jpg', 'frame_0302.jpg']  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   33 / 77

--- End of Page 33 ---

In [39]:   gray_frames = [] for frame_file in frame_files:      path = os.path.join(          sample_video_dir,          frame_file      )      img = Image.open(path).convert("L")      gray_frames.append(img) frame_differences = [] for i in range(1, 5):      current = np.asarray(          gray_frames[i]      ).astype(np.float32)      previous = np.asarray(          gray_frames[i-1]      ).astype(np.float32)      diff = np.abs(          current - previous      )      diff = np.clip(          diff, 0, 255      ).astype(np.uint8)      frame_differences.append(          Image.fromarray(diff)      ) print("Frame differences:", len(frame_differences))  Frame differences: 4  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   34 / 77

--- End of Page 34 ---

In [40]:   batch_tensors = torch.stack([      gray_transform(img)      for img in frame_differences ]).to(device) print("Batch:", batch_tensors.shape) with torch.no_grad():      batch_features = temporal_efficientnet(          batch_tensors      ) print("Features:", batch_features.shape)  Batch: torch.Size([4, 3, 224, 224]) Features: torch.Size([4, 1792])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   35 / 77

--- End of Page 35 ---

In [41]:   import os import numpy as np import pandas as pd import torch from PIL import Image from tqdm import tqdm TEMPORAL_FAST_ROOT = "/kaggle/working/temporal_cnn_features_fast" CHECKPOINT = "/kaggle/working/temporal_cnn_fast_checkpoint.csv" os.makedirs(TEMPORAL_FAST_ROOT, exist_ok=True) # Resume support if os.path.exists(CHECKPOINT):      ckpt = pd.read_csv(CHECKPOINT)      processed = set(ckpt["video_id"].astype(str))      results = ckpt.to_dict("records")      print("Resuming:", len(processed), "videos") else:      processed = set()      results = []      print("Starting fresh.") temporal_efficientnet.eval() SAVE_EVERY = 100 count = 0 for label, label_dir in [      ("Real", REAL_DIR),      ("Fake", FAKE_DIR) ]:      video_ids = sorted(os.listdir(label_dir))      for video_id in tqdm(          video_ids,          desc=f"Temporal CNN {label}"      ):          if video_id in processed:              continue          video_dir = os.path.join(              label_dir,              video_id          )          if not os.path.isdir(video_dir):              continue          frame_files = sorted([              f for f in os.listdir(video_dir)              if f.lower().endswith(".jpg")          ])          if len(frame_files) != 5:  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   36 / 77

--- End of Page 36 ---

results.append({                  "video_id": video_id,                  "label": label,                  "status": "skipped"              })              processed.add(video_id)              continue          try:              # -------------------------------              # Load 5 grayscale frames              # -------------------------------              gray_frames = []              for frame_file in frame_files:                  path = os.path.join(                      video_dir,                      frame_file                  )                  img = Image.open(                      path                  ).convert("L")                  gray_frames.append(img)              # -------------------------------              # Create 4 differences              # -------------------------------              differences = []              for i in range(1, 5):                  current = np.asarray(                      gray_frames[i]                  ).astype(np.float32)                  previous = np.asarray(                      gray_frames[i-1]                  ).astype(np.float32)                  diff = np.abs(                      current - previous                  )                  diff = np.clip(                      diff,                      0,                      255                  ).astype(np.uint8)                  differences.append(                      Image.fromarray(diff)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   37 / 77

--- End of Page 37 ---

)              # -------------------------------              # Batch → EfficientNet-B4              # -------------------------------              batch = torch.stack([                  gray_transform(img)                  for img in differences              ]).to(device)              with torch.no_grad():                  features = temporal_efficientnet(                      batch                  )              features = (                  features                  .cpu()                  .numpy()              )              # Shape: (4, 1792)              # -------------------------------              # Save              # -------------------------------              feature_path = os.path.join(                  TEMPORAL_FAST_ROOT,                  f"{label}_{video_id}.npy"              )              np.save(                  feature_path,                  features              )              results.append({                  "video_id": video_id,                  "label": label,                  "feature_shape": "4x1792",                  "feature_path": feature_path,                  "status": "processed"              })              processed.add(video_id)              count += 1              # -------------------------------              # Checkpoint              # -------------------------------              if count % SAVE_EVERY == 0:                  pd.DataFrame(  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   38 / 77

--- End of Page 38 ---

results                  ).to_csv(                      CHECKPOINT,                      index=False                  )                  print(                      f"\nCheckpoint saved: "                      f"{len(processed)} videos"                  )          except Exception as e:              print(                  f"\nError in {video_id}: {e}"              )              results.append({                  "video_id": video_id,                  "label": label,                  "status": f"error: {e}"              })              processed.add(video_id) # Final checkpoint pd.DataFrame(      results ).to_csv(      CHECKPOINT,      index=False ) print("\n==============================") print("TEMPORAL CNN COMPLETE") print("==============================") print(      "Videos processed:",      len(processed) ) print(      "Output:",      TEMPORAL_FAST_ROOT )  Starting fresh. Temporal CNN Real:  12%|█ ▏         | 103/888 [00:03<00:27,   28.92it/s] Checkpoint saved: 100 videos  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   39 / 77

--- End of Page 39 ---

Temporal CNN Real:  23%|██ ▎        | 206/888 [00:07<00:22,   29.78it/s] Checkpoint saved: 202 videos Temporal CNN Real:  35%|███ ▍       | 307/888 [00:10<00:20,   28.75it/s] Checkpoint saved: 302 videos Temporal CNN Real:  46%|████ ▋      | 411/888 [00:14<00:16,   29.32it/s] Checkpoint saved: 408 videos Temporal CNN Real:  58%|█████ ▊     | 513/888 [00:17<00:12,   29.94it/s] Checkpoint saved: 508 videos Temporal CNN Real:  69%|██████ ▉    | 613/888 [00:20<00:09,   29.07it/s] Checkpoint saved: 608 videos Temporal CNN Real:  80%|████████  | 711/888 [00:24<00:06,   28.94it/s] Checkpoint saved: 708 videos Temporal CNN Real:  92%|█████████ ▏ | 814/888 [00:27<00:02,   28.63it/s] Checkpoint saved: 808 videos Temporal CNN Real: 100%|██████████| 888/888 [00:30<00:00,   29.18it/s] Temporal CNN Fake:   0%|          | 25/5639 [00:00<03:09,   29.55it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   40 / 77

--- End of Page 40 ---

Checkpoint saved: 909 videos Temporal CNN Fake:   2%| ▏          | 127/5639 [00:04<03:04,   29.94it/s] Checkpoint saved: 1009 videos Temporal CNN Fake:   4%| ▍          | 223/5639 [00:07<02:59,   30.10it/s] Checkpoint saved: 1109 videos Temporal CNN Fake:   6%|▌         | 326/5639 [00:10<02:59,   29.64it/s] Checkpoint saved: 1209 videos Temporal CNN Fake:   8%| ▊          | 425/5639 [00:14<02:58,   29.28it/s] Checkpoint saved: 1309 videos Temporal CNN Fake:   9%| ▉          | 527/5639 [00:17<02:53,   29.53it/s] Checkpoint saved: 1409 videos Temporal CNN Fake:  11%|█         | 625/5639 [00:21<02:52,   29.02it/s] Checkpoint saved: 1509 videos Temporal CNN Fake:  13%|█ ▎         | 724/5639 [00:24<02:46,   29.57it/s] Checkpoint saved: 1609 videos Temporal CNN Fake:  15%|█ ▍         | 826/5639 [00:27<02:48,   28.63it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   41 / 77

--- End of Page 41 ---

Checkpoint saved: 1709 videos Temporal CNN Fake:  16%|█ ▋         | 925/5639 [00:31<02:36,   30.19it/s] Checkpoint saved: 1809 videos Temporal CNN Fake:  18%|█ ▊         | 1027/5639 [00:34<02:32,   30.18it/s] Checkpoint saved: 1909 videos Temporal CNN Fake:  20%|█ ▉         | 1125/5639 [00:37<02:28,   30.41it/s] Checkpoint saved: 2009 videos Temporal CNN Fake:  22%|██ ▏        | 1225/5639 [00:41<02:33,   28.69it/s] Checkpoint saved: 2109 videos Temporal CNN Fake:  24%|██ ▎        | 1327/5639 [00:44<02:25,   29.62it/s] Checkpoint saved: 2209 videos Temporal CNN Fake:  25%|██▌       | 1426/5639 [00:48<02:21,   29.72it/s] Checkpoint saved: 2309 videos Temporal CNN Fake:  27%|██ ▋        | 1525/5639 [00:51<02:18,   29.67it/s] Checkpoint saved: 2409 videos Temporal CNN Fake:  29%|██ ▉        | 1625/5639 [00:54<02:15,   29.62it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   42 / 77

--- End of Page 42 ---

Checkpoint saved: 2509 videos Temporal CNN Fake:  31%|███       | 1724/5639 [00:58<02:12,   29.48it/s] Checkpoint saved: 2609 videos Temporal CNN Fake:  32%|███ ▏       | 1823/5639 [01:01<02:11,   29.00it/s] Checkpoint saved: 2709 videos Temporal CNN Fake:  34%|███ ▍       | 1924/5639 [01:04<02:07,   29.14it/s] Checkpoint saved: 2809 videos Temporal CNN Fake:  36%|███▌      | 2023/5639 [01:08<02:04,   29.12it/s] Checkpoint saved: 2909 videos Temporal CNN Fake:  38%|███ ▊       | 2126/5639 [01:11<02:00,   29.09it/s] Checkpoint saved: 3009 videos Temporal CNN Fake:  39%|███ ▉       | 2225/5639 [01:14<02:00,   28.39it/s] Checkpoint saved: 3109 videos Temporal CNN Fake:  41%|████      | 2325/5639 [01:18<01:52,   29.51it/s] Checkpoint saved: 3209 videos Temporal CNN Fake:  43%|████ ▎      | 2424/5639 [01:21<01:53,   28.36it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   43 / 77

--- End of Page 43 ---

Checkpoint saved: 3309 videos Temporal CNN Fake:  45%|████ ▍      | 2527/5639 [01:25<01:42,   30.45it/s] Checkpoint saved: 3409 videos Temporal CNN Fake:  47%|████ ▋      | 2627/5639 [01:28<01:39,   30.18it/s] Checkpoint saved: 3509 videos Temporal CNN Fake:  48%|████ ▊      | 2724/5639 [01:31<01:40,   28.92it/s] Checkpoint saved: 3609 videos Temporal CNN Fake:  50%|█████     | 2825/5639 [01:35<01:35,   29.48it/s] Checkpoint saved: 3709 videos Temporal CNN Fake:  52%|█████ ▏     | 2924/5639 [01:38<01:33,   28.99it/s] Checkpoint saved: 3809 videos Temporal CNN Fake:  54%|█████ ▎     | 3025/5639 [01:41<01:32,   28.26it/s] Checkpoint saved: 3909 videos Temporal CNN Fake:  55%|█████▌    | 3127/5639 [01:45<01:23,   29.94it/s] Checkpoint saved: 4009 videos Temporal CNN Fake:  57%|█████ ▋     | 3225/5639 [01:48<01:22,   29.19it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   44 / 77

--- End of Page 44 ---

Checkpoint saved: 4109 videos Temporal CNN Fake:  59%|█████ ▉     | 3324/5639 [01:51<01:28,   26.27it/s] Checkpoint saved: 4209 videos Temporal CNN Fake:  61%|██████    | 3426/5639 [01:55<01:17,   28.45it/s] Checkpoint saved: 4309 videos Temporal CNN Fake:  63%|██████ ▎    | 3526/5639 [01:58<01:10,   30.13it/s] Checkpoint saved: 4411 videos Temporal CNN Fake:  64%|██████ ▍    | 3627/5639 [02:02<01:08,   29.21it/s] Checkpoint saved: 4512 videos Temporal CNN Fake:  66%|██████▌   | 3729/5639 [02:05<01:06,   28.93it/s] Checkpoint saved: 4612 videos Temporal CNN Fake:  68%|██████ ▊    | 3827/5639 [02:08<01:02,   28.77it/s] Checkpoint saved: 4712 videos Temporal CNN Fake:  70%|██████ ▉    | 3929/5639 [02:12<00:59,   28.72it/s] Checkpoint saved: 4812 videos Temporal CNN Fake:  71%|███████ ▏   | 4029/5639 [02:15<00:55,   29.26it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   45 / 77

--- End of Page 45 ---

Checkpoint saved: 4912 videos Temporal CNN Fake:  73%|███████ ▎   | 4129/5639 [02:18<00:50,   30.04it/s] Checkpoint saved: 5012 videos Temporal CNN Fake:  75%|███████ ▍   | 4228/5639 [02:22<00:47,   30.01it/s] Checkpoint saved: 5112 videos Temporal CNN Fake:  77%|███████ ▋   | 4328/5639 [02:25<00:47,   27.74it/s] Checkpoint saved: 5212 videos Temporal CNN Fake:  79%|███████ ▊   | 4429/5639 [02:28<00:40,   29.82it/s] Checkpoint saved: 5312 videos Temporal CNN Fake:  80%|████████  | 4527/5639 [02:32<00:38,   29.13it/s] Checkpoint saved: 5412 videos Temporal CNN Fake:  82%|████████ ▏  | 4627/5639 [02:35<00:34,   29.07it/s] Checkpoint saved: 5512 videos Temporal CNN Fake:  84%|████████ ▍  | 4728/5639 [02:38<00:31,   29.18it/s] Checkpoint saved: 5612 videos Temporal CNN Fake:  86%|████████▌ | 4827/5639 [02:42<00:30,   26.49it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   46 / 77

--- End of Page 46 ---

Checkpoint saved: 5712 videos Temporal CNN Fake:  87%|████████ ▋  | 4930/5639 [02:45<00:24,   29.41it/s] Checkpoint saved: 5812 videos Temporal CNN Fake:  89%|████████ ▉  | 5032/5639 [02:48<00:20,   30.21it/s] Checkpoint saved: 5914 videos Temporal CNN Fake:  91%|█████████ | 5132/5639 [02:52<00:17,   28.98it/s] Checkpoint saved: 6014 videos Temporal CNN Fake:  93%|█████████ ▎ | 5231/5639 [02:55<00:14,   28.96it/s] Checkpoint saved: 6114 videos Temporal CNN Fake:  95%|█████████ ▍ | 5330/5639 [02:58<00:11,   26.60it/s] Checkpoint saved: 6214 videos Temporal CNN Fake:  96%|█████████ ▋ | 5432/5639 [03:02<00:07,   29.36it/s] Checkpoint saved: 6314 videos Temporal CNN Fake:  98%|█████████ ▊ | 5531/5639 [03:05<00:03,   29.62it/s] Checkpoint saved: 6414 videos Temporal CNN Fake: 100%|█████████ ▉ | 5631/5639 [03:08<00:00,   28.50it/s]  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   47 / 77

--- End of Page 47 ---

Checkpoint saved: 6514 videos Temporal CNN Fake: 100%|██████████| 5639/5639 [03:09<00:00,   29.80it/s] ============================== TEMPORAL CNN COMPLETE ============================== Videos processed: 6527 Output: /kaggle/working/temporal_cnn_features_fast  In [42]:   files = [      f for f in os.listdir(TEMPORAL_FAST_ROOT)      if f.endswith(".npy") ] print("Feature files:", len(files)) test = np.load(      os.path.join(          TEMPORAL_FAST_ROOT,          files[0]      ) ) print("Feature shape:", test.shape)  Feature files: 6513 Feature shape: (4, 1792)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   48 / 77

--- End of Page 48 ---

In [43]:   print("Total processed records:", len(processed)) print("Feature files:", len(files)) skipped = [      r for r in results      if r.get("status") == "skipped" ] print("Skipped videos:", len(skipped)) print("Skipped IDs:") print([r["video_id"] for r in skipped[:20]])  Total processed records: 6527 Feature files: 6513 Skipped videos: 14 Skipped IDs: ['00128', '00187', 'id10_0001', 'id10_0003', 'id10_0006',   'id10_0007', 'id11_0000', 'id1_0000', 'id8_0009',   'id39_id40_0005', 'id39_id42_0005', 'id39_id45_0005',   'id55_id50_0004', 'id55_id53_0004']  In [44]:   print("Total processed records:", len(processed)) print("Feature files:", len(files)) skipped = [      r for r in results      if r.get("status") == "skipped" ] print("Skipped videos:", len(skipped)) print("Skipped IDs:") print([r["video_id"] for r in skipped[:20]])  Total processed records: 6527 Feature files: 6513 Skipped videos: 14 Skipped IDs: ['00128', '00187', 'id10_0001', 'id10_0003', 'id10_0006',   'id10_0007', 'id11_0000', 'id1_0000', 'id8_0009',   'id39_id40_0005', 'id39_id42_0005', 'id39_id45_0005',   'id55_id50_0004', 'id55_id53_0004']  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   49 / 77

--- End of Page 49 ---

In [45]:   import shutil import os TEMPORAL_ZIP_BASE = "/kaggle/working/temporal_cnn_features_fast" shutil.make_archive(      TEMPORAL_ZIP_BASE,      "zip",      TEMPORAL_FAST_ROOT ) TEMPORAL_ZIP = TEMPORAL_ZIP_BASE + ".zip" print("ZIP created:", os.path.exists(TEMPORAL_ZIP)) print(      "ZIP size (MB):",      round(          os.path.getsize(TEMPORAL_ZIP) / (1024**2),          2      ) )  ZIP created: True ZIP size (MB): 164.19  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   50 / 77

--- End of Page 50 ---

In [47]:   import os SPATIAL_ROOT = "/kaggle/input/datasets/srujanamannepuli/spatial" TEMPORAL_ROOT = "/kaggle/input/datasets/srujanamannepuli/temporal" spatial_files = sorted([      f for f in os.listdir(SPATIAL_ROOT)      if f.endswith(".npy") ]) temporal_files = sorted([      f for f in os.listdir(TEMPORAL_ROOT)      if f.endswith(".npy") ]) print("Spatial feature files:", len(spatial_files)) print("Temporal feature files:", len(temporal_files)) print("\nSpatial samples:") print(spatial_files[:5]) print("\nTemporal samples:") print(temporal_files[:5])  Spatial feature files: 6513 Temporal feature files: 6513 Spatial samples: ['Fake_id0_id16_0000.npy', 'Fake_id0_id16_0001.npy',   'Fake_id0_id16_0002.npy', 'Fake_id0_id16_0003.npy',   'Fake_id0_id16_0004.npy'] Temporal samples: ['Fake_id0_id16_0000.npy', 'Fake_id0_id16_0001.npy',   'Fake_id0_id16_0002.npy', 'Fake_id0_id16_0003.npy',   'Fake_id0_id16_0004.npy']  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   51 / 77

--- End of Page 51 ---

In [48]:   import numpy as np import os s = np.load(      os.path.join(          SPATIAL_ROOT,          spatial_files[0]      ) ) t = np.load(      os.path.join(          TEMPORAL_ROOT,          temporal_files[0]      ) ) print("Spatial shape:", s.shape) print("Temporal shape:", t.shape)  Spatial shape: (5, 3840) Temporal shape: (4, 1792)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   52 / 77

--- End of Page 52 ---

In [49]:   import os import pandas as pd spatial_map = {      os.path.splitext(f)[0]: os.path.join(SPATIAL_ROOT, f)      for f in spatial_files } temporal_map = {      os.path.splitext(f)[0]: os.path.join(TEMPORAL_ROOT, f)      for f in temporal_files } pairs = [] for key in sorted(      set(spatial_map.keys()) &      set(temporal_map.keys()) ):      # Filename format: Real_videoID / Fake_videoID      if key.startswith("Real_"):          label = 0      elif key.startswith("Fake_"):          label = 1      else:          continue      pairs.append({          "video_key": key,          "label": label,          "spatial_path": spatial_map[key],          "temporal_path": temporal_map[key]      }) pairs_df = pd.DataFrame(pairs) print("Paired videos:", len(pairs_df)) print("\nClass distribution:") print(pairs_df["label"].value_counts())  Paired videos: 6513 Class distribution: label 1    5634 0     879 Name: count, dtype: int64  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   53 / 77

--- End of Page 53 ---

In [51]:   from sklearn.model_selection import train_test_split train_df, temp_df = train_test_split(      pairs_df,      test_size=0.30,      stratify=pairs_df["label"],      random_state=42 ) val_df, test_df = train_test_split(      temp_df,      test_size=0.50,      stratify=temp_df["label"],      random_state=42 ) print("Train:", len(train_df)) print("Validation:", len(val_df)) print("Test:", len(test_df)) print("\nTrain distribution:") print(train_df["label"].value_counts()) print("\nValidation distribution:") print(val_df["label"].value_counts()) print("\nTest distribution:") print(test_df["label"].value_counts())  Train: 4559 Validation: 977 Test: 977 Train distribution: label 1    3944 0     615 Name: count, dtype: int64 Validation distribution: label 1    845 0    132 Name: count, dtype: int64 Test distribution: label 1    845 0    132 Name: count, dtype: int64  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   54 / 77

--- End of Page 54 ---

In [53]:   import torch from torch.utils.data import Dataset, DataLoader import numpy as np class DeepfakeFeatureDataset(Dataset):      def __init__(self, dataframe):          self.df = dataframe.reset_index(drop=True)      def __len__(self):          return len(self.df)      def __getitem__(self, idx):          row = self.df.iloc[idx]          spatial = np.load(              row["spatial_path"]          ).astype(np.float32)          temporal = np.load(              row["temporal_path"]          ).astype(np.float32)          label = np.int64(              row["label"]          )          return (              torch.from_numpy(spatial),              torch.from_numpy(temporal),              torch.tensor(                  label,                  dtype=torch.long              )          )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   55 / 77

--- End of Page 55 ---

In [55]:   train_dataset = DeepfakeFeatureDataset(      train_df ) val_dataset = DeepfakeFeatureDataset(      val_df ) test_dataset = DeepfakeFeatureDataset(      test_df ) print("Train:", len(train_dataset)) print("Validation:", len(val_dataset)) print("Test:", len(test_dataset))  Train: 4559 Validation: 977 Test: 977  In [56]:   train_loader = DataLoader(      train_dataset,      batch_size=16,      shuffle=True,      num_workers=2,      pin_memory=True ) val_loader = DataLoader(      val_dataset,      batch_size=16,      shuffle=False,      num_workers=2,      pin_memory=True ) test_loader = DataLoader(      test_dataset,      batch_size=16,      shuffle=False,      num_workers=2,      pin_memory=True ) print("Train batches:", len(train_loader)) print("Validation batches:", len(val_loader)) print("Test batches:", len(test_loader))  Train batches: 285 Validation batches: 62 Test batches: 62  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   56 / 77

--- End of Page 56 ---

In [57]:   spatial_batch, temporal_batch, labels = next(      iter(train_loader) ) print("Spatial batch:", spatial_batch.shape) print("Temporal batch:", temporal_batch.shape) print("Labels:", labels.shape)  Spatial batch: torch.Size([16, 5, 3840]) Temporal batch: torch.Size([16, 4, 1792]) Labels: torch.Size([16])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   57 / 77

--- End of Page 57 ---

In [58]:   import torch import torch.nn as nn import torch.nn.functional as F class DeepfakeFusionModel(nn.Module):      def __init__(          self,          spatial_dim=3840,          temporal_dim=1792,          hidden_size=256,          num_heads=8      ):          super().__init__()          # -----------------------------          # Spatial Projection          # 3840 → 512          # -----------------------------          self.spatial_projection = nn.Linear(              spatial_dim,              512          )          # -----------------------------          # Temporal BiLSTM          # 1792 → 512          # -----------------------------          self.temporal_bilstm = nn.LSTM(              input_size=temporal_dim,              hidden_size=hidden_size,              num_layers=2,              batch_first=True,              bidirectional=True          )          # -----------------------------          # Cross Attention          # Q = Spatial          # K,V = Temporal          # -----------------------------          self.q_projection = nn.Linear(              512, 512          )          self.k_projection = nn.Linear(              512, 512          )          self.v_projection = nn.Linear(              512, 512          )          self.cross_attention = nn.MultiheadAttention(              embed_dim=512,              num_heads=num_heads,  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   58 / 77

--- End of Page 58 ---

batch_first=True          )          self.layer_norm = nn.LayerNorm(512)          # -----------------------------          # Adaptive Gating          # -----------------------------          self.gate = nn.Sequential(              nn.Linear(                  1024,                  512              ),              nn.Sigmoid()          )          # -----------------------------          # Classification Head          # -----------------------------          self.fc1 = nn.Linear(              512,              256          )          self.dropout = nn.Dropout(0.5)          self.fc2 = nn.Linear(              256,              128          )          self.fc3 = nn.Linear(              128,              2          )      def forward(          self,          spatial,          temporal      ):          # spatial:          # (B, 5, 3840)          # temporal:          # (B, 4, 1792)          # =============================          # Spatial Branch          # =============================          spatial = self.spatial_projection(              spatial          )          # (B, 5, 512)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   59 / 77

--- End of Page 59 ---

# =============================          # Temporal Branch          # =============================          temporal, _ = self.temporal_bilstm(              temporal          )          # (B, 4, 512)          # =============================          # Cross Attention          # =============================          Q = self.q_projection(              spatial          )          K = self.k_projection(              temporal          )          V = self.v_projection(              temporal          )          attention_output, attention_weights = (              self.cross_attention(                  Q,                  K,                  V              )          )          # (B, 5, 512)          # =============================          # Layer Normalization          # =============================          attention_output = self.layer_norm(              spatial + attention_output          )          # =============================          # Adaptive Gating          # =============================          spatial_global = spatial.mean(              dim=1          )          attention_global = attention_output.mean(              dim=1          )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   60 / 77

--- End of Page 60 ---

gate_input = torch.cat(              [                  spatial_global,                  attention_global              ],              dim=1          )          gate = self.gate(              gate_input          )          fused = (              gate * attention_global              +              (1 - gate) * spatial_global          )          # (B, 512)          # =============================          # Classification          # =============================          x = F.relu(              self.fc1(fused)          )          x = self.dropout(x)          x = F.relu(              self.fc2(x)          )          logits = self.fc3(x)          probabilities = F.softmax(              logits,              dim=1          )          return logits, probabilities, attention_weights  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   61 / 77

--- End of Page 61 ---

In [59]:   model = DeepfakeFusionModel(      spatial_dim=3840,      temporal_dim=1792,      hidden_size=256,      num_heads=8 ).to(device) print(model)  DeepfakeFusionModel(    (spatial_projection): Linear(in_features=3840, out_features=512,   bias=True)    (temporal_bilstm): LSTM(1792, 256, num_layers=2,   batch_first=True, bidirectional=True)    (q_projection): Linear(in_features=512, out_features=512,   bias=True)    (k_projection): Linear(in_features=512, out_features=512,   bias=True)    (v_projection): Linear(in_features=512, out_features=512,   bias=True)    (cross_attention): MultiheadAttention(      (out_proj): NonDynamicallyQuantizableLinear(in_features=512,   out_features=512, bias=True)    )    (layer_norm): LayerNorm((512,), eps=1e-05,   elementwise_affine=True)    (gate): Sequential(      (0): Linear(in_features=1024, out_features=512, bias=True)      (1): Sigmoid()    )    (fc1): Linear(in_features=512, out_features=256, bias=True)    (dropout): Dropout(p=0.5, inplace=False)    (fc2): Linear(in_features=256, out_features=128, bias=True)    (fc3): Linear(in_features=128, out_features=2, bias=True) )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   62 / 77

--- End of Page 62 ---

In [60]:   criterion = nn.CrossEntropyLoss() optimizer = torch.optim.Adam(      model.parameters(),      lr=1e-4,      weight_decay=1e-5 ) print("Optimizer:", optimizer) print("Loss:", criterion)  Optimizer: Adam ( Parameter Group 0      amsgrad: False      betas: (0.9, 0.999)      capturable: False      decoupled_weight_decay: False      differentiable: False      eps: 1e-08      foreach: None      fused: None      lr: 0.0001      maximize: False      weight_decay: 1e-05 ) Loss: CrossEntropyLoss()  In [61]:   model.eval() with torch.no_grad():      spatial_batch = spatial_batch.to(device)      temporal_batch = temporal_batch.to(device)      logits, probabilities, attention = model(          spatial_batch,          temporal_batch      ) print("Spatial input:", spatial_batch.shape) print("Temporal input:", temporal_batch.shape) print("Logits:", logits.shape) print("Probabilities:", probabilities.shape) print("Attention:", attention.shape)  Spatial input: torch.Size([16, 5, 3840]) Temporal input: torch.Size([16, 4, 1792]) Logits: torch.Size([16, 2]) Probabilities: torch.Size([16, 2]) Attention: torch.Size([16, 5, 4])  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   63 / 77

--- End of Page 63 ---

In [62]:   print(probabilities[:5]) print(      "Probability sums:",      probabilities[:5].sum(dim=1) )  tensor([[0.5114, 0.4886],          [0.5099, 0.4901],          [0.5003, 0.4997],          [0.5044, 0.4956],          [0.5034, 0.4966]], device='cuda:0') Probability sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000],   device='cuda:0')  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   64 / 77

--- End of Page 64 ---

In [63]:   from sklearn.metrics import (      accuracy_score,      precision_score,      recall_score,      f1_score,      roc_auc_score ) def train_one_epoch(model, loader, optimizer, criterion, device):      model.train()      running_loss = 0.0      all_labels = []      all_preds = []      for spatial, temporal, labels in loader:          spatial = spatial.to(device, non_blocking=True)          temporal = temporal.to(device, non_blocking=True)          labels = labels.to(device, non_blocking=True)          optimizer.zero_grad()          logits, probabilities, _ = model(              spatial,              temporal          )          loss = criterion(              logits,              labels          )          loss.backward()          optimizer.step()          running_loss += (              loss.item() * labels.size(0)          )          preds = torch.argmax(              logits,              dim=1          )          all_labels.extend(              labels.detach().cpu().numpy()          )          all_preds.extend(              preds.detach().cpu().numpy()          )      epoch_loss = (          running_loss / len(loader.dataset)  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   65 / 77

--- End of Page 65 ---

)      accuracy = accuracy_score(          all_labels,          all_preds      )      return epoch_loss, accuracy  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   66 / 77

--- End of Page 66 ---

In [64]:   def validate(      model,      loader,      criterion,      device ):      model.eval()      running_loss = 0.0      all_labels = []      all_preds = []      all_probs = []      with torch.no_grad():          for spatial, temporal, labels in loader:              spatial = spatial.to(                  device,                  non_blocking=True              )              temporal = temporal.to(                  device,                  non_blocking=True              )              labels = labels.to(                  device,                  non_blocking=True              )              logits, probabilities, _ = model(                  spatial,                  temporal              )              loss = criterion(                  logits,                  labels              )              running_loss += (                  loss.item() *                  labels.size(0)              )              preds = torch.argmax(                  logits,                  dim=1              )              all_labels.extend(                  labels.cpu().numpy()              )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   67 / 77

--- End of Page 67 ---

all_preds.extend(                  preds.cpu().numpy()              )              all_probs.extend(                  probabilities[:, 1]                  .cpu()                  .numpy()              )      epoch_loss = (          running_loss /          len(loader.dataset)      )      accuracy = accuracy_score(          all_labels,          all_preds      )      precision = precision_score(          all_labels,          all_preds,          zero_division=0      )      recall = recall_score(          all_labels,          all_preds,          zero_division=0      )      f1 = f1_score(          all_labels,          all_preds,          zero_division=0      )      auc = roc_auc_score(          all_labels,          all_probs      )      return (          epoch_loss,          accuracy,          precision,          recall,          f1,          auc      )  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   68 / 77

--- End of Page 68 ---

In [65]:   NUM_EPOCHS = 5 best_f1 = 0.0 history = [] for epoch in range(NUM_EPOCHS):      train_loss, train_acc = train_one_epoch(          model,          train_loader,          optimizer,          criterion,          device      )      (          val_loss,          val_acc,          val_precision,          val_recall,          val_f1,          val_auc      ) = validate(          model,          val_loader,          criterion,          device      )      print(          f"\nEpoch {epoch + 1}/{NUM_EPOCHS}"      )      print(          f"Train Loss: {train_loss:.4f} | "          f"Train Acc: {train_acc:.4f}"      )      print(          f"Val Loss: {val_loss:.4f} | "          f"Val Acc: {val_acc:.4f}"      )      print(          f"Precision: {val_precision:.4f} | "          f"Recall: {val_recall:.4f} | "          f"F1: {val_f1:.4f} | "          f"AUC: {val_auc:.4f}"      )      history.append({          "epoch": epoch + 1,          "train_loss": train_loss,          "train_accuracy": train_acc,          "val_loss": val_loss,          "val_accuracy": val_acc,  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   69 / 77

--- End of Page 69 ---

"precision": val_precision,          "recall": val_recall,          "f1": val_f1,          "auc": val_auc      })      # Save best model      if val_f1 > best_f1:          best_f1 = val_f1          torch.save(              {                  "epoch": epoch + 1,                  "model_state_dict": model.state_dict(),                  "optimizer_state_dict": optimizer.state_dict(),                  "val_f1": val_f1,                  "val_auc": val_auc              },              "/kaggle/working/best_deepfake_model.pth"          )          print(" ✅  Best model saved!")  Epoch 1/5 Train Loss: 0.3919 | Train Acc: 0.8638 Val Loss: 0.3937 | Val Acc: 0.8649 Precision: 0.8649 | Recall: 1.0000 | F1: 0.9276 | AUC: 0.7251  ✅  Best model saved! Epoch 2/5 Train Loss: 0.3317 | Train Acc: 0.8754 Val Loss: 0.3297 | Val Acc: 0.8751 Precision: 0.8858 | Recall: 0.9822 | F1: 0.9315 | AUC: 0.7885  ✅  Best model saved! Epoch 3/5 Train Loss: 0.2949 | Train Acc: 0.8905 Val Loss: 0.3079 | Val Acc: 0.8864 Precision: 0.8888 | Recall: 0.9929 | F1: 0.9380 | AUC: 0.8188  ✅  Best model saved! Epoch 4/5 Train Loss: 0.2508 | Train Acc: 0.9044 Val Loss: 0.2988 | Val Acc: 0.8823 Precision: 0.9129 | Recall: 0.9550 | F1: 0.9335 | AUC: 0.8329 Epoch 5/5 Train Loss: 0.2346 | Train Acc: 0.9129 Val Loss: 0.2980 | Val Acc: 0.8976 Precision: 0.9001 | Recall: 0.9917 | F1: 0.9437 | AUC: 0.8327  ✅  Best model saved!  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   70 / 77

--- End of Page 70 ---

In [67]:   checkpoint = torch.load(      "/kaggle/working/best_deepfake_model.pth",      map_location=device,      weights_only=False ) model.load_state_dict(      checkpoint["model_state_dict"] ) optimizer.load_state_dict(      checkpoint["optimizer_state_dict"] ) start_epoch = checkpoint["epoch"] best_f1 = checkpoint["val_f1"] print("Resuming from epoch:", start_epoch) print("Best validation F1:", checkpoint["val_f1"]) print("Best validation AUC:", checkpoint["val_auc"])  Resuming from epoch: 5 Best validation F1: 0.9436936936936937 Best validation AUC: 0.8327057557826789  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   71 / 77

--- End of Page 71 ---

In [74]:   TOTAL_EPOCHS = 20 for epoch in range(      start_epoch,      TOTAL_EPOCHS ):      train_loss, train_acc = train_one_epoch(          model,          train_loader,          optimizer,          criterion,          device      )      (          val_loss,          val_acc,          val_precision,          val_recall,          val_f1,          val_auc      ) = validate(          model,          val_loader,          criterion,          device      )      print(          f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS}"      )      print(          f"Train Loss: {train_loss:.4f} | "          f"Train Acc: {train_acc:.4f}"      )      print(          f"Val Loss: {val_loss:.4f} | "          f"Val Acc: {val_acc:.4f}"      )      print(          f"Precision: {val_precision:.4f} | "          f"Recall: {val_recall:.4f} | "          f"F1: {val_f1:.4f} | "          f"AUC: {val_auc:.4f}"      )      history.append({          "epoch": epoch + 1,          "train_loss": train_loss,          "train_accuracy": train_acc,          "val_loss": val_loss,          "val_accuracy": val_acc,          "precision": val_precision,  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   72 / 77

--- End of Page 72 ---

"recall": val_recall,          "f1": val_f1,          "auc": val_auc      })      if val_f1 > best_f1:          best_f1 = val_f1          torch.save(              {                  "epoch": epoch + 1,                  "model_state_dict": model.state_dict(),                  "optimizer_state_dict": optimizer.state_dict(),                  "val_f1": val_f1,                  "val_auc": val_auc              },              "/kaggle/working/best_deepfake_model.pth"          )          print(" 🔥  New best model saved!")  Epoch 6/20 Train Loss: 0.0653 | Train Acc: 0.9778 Val Loss: 0.4602 | Val Acc: 0.8997 Precision: 0.9100 | Recall: 0.9811 | F1: 0.9442 | AUC: 0.8463 Epoch 7/20 Train Loss: 0.0646 | Train Acc: 0.9781 Val Loss: 0.4774 | Val Acc: 0.9007 Precision: 0.9193 | Recall: 0.9704 | F1: 0.9442 | AUC: 0.8413 Epoch 8/20 Train Loss: 0.0529 | Train Acc: 0.9811 Val Loss: 0.5432 | Val Acc: 0.8997 Precision: 0.9029 | Recall: 0.9905 | F1: 0.9447 | AUC: 0.8303 Epoch 9/20 Train Loss: 0.0382 | Train Acc: 0.9862 Val Loss: 0.5082 | Val Acc: 0.8997 Precision: 0.9240 | Recall: 0.9633 | F1: 0.9432 | AUC: 0.8377 Epoch 10/20 Train Loss: 0.0572 | Train Acc: 0.9796 Val Loss: 0.5806 | Val Acc: 0.9007 Precision: 0.9092 | Recall: 0.9834 | F1: 0.9449 | AUC: 0.8393 Epoch 11/20 Train Loss: 0.0297 | Train Acc: 0.9897 Val Loss: 0.5942 | Val Acc: 0.9058 Precision: 0.9179 | Recall: 0.9787 | F1: 0.9473 | AUC: 0.8429 Epoch 12/20 Train Loss: 0.0316 | Train Acc: 0.9888 Val Loss: 0.5477 | Val Acc: 0.9038 Precision: 0.9262 | Recall: 0.9657 | F1: 0.9455 | AUC: 0.8514  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   73 / 77

--- End of Page 73 ---

Epoch 13/20 Train Loss: 0.0437 | Train Acc: 0.9835 Val Loss: 0.6004 | Val Acc: 0.8966 Precision: 0.9143 | Recall: 0.9716 | F1: 0.9421 | AUC: 0.8379 Epoch 14/20 Train Loss: 0.0284 | Train Acc: 0.9899 Val Loss: 0.6058 | Val Acc: 0.8915 Precision: 0.9156 | Recall: 0.9633 | F1: 0.9389 | AUC: 0.8408 Epoch 15/20 Train Loss: 0.0222 | Train Acc: 0.9912 Val Loss: 0.6455 | Val Acc: 0.8936 Precision: 0.9158 | Recall: 0.9657 | F1: 0.9401 | AUC: 0.8381 Epoch 16/20 Train Loss: 0.0293 | Train Acc: 0.9908 Val Loss: 0.7037 | Val Acc: 0.8987 Precision: 0.9063 | Recall: 0.9846 | F1: 0.9438 | AUC: 0.8265 Epoch 17/20 Train Loss: 0.0293 | Train Acc: 0.9888 Val Loss: 0.4856 | Val Acc: 0.8966 Precision: 0.9133 | Recall: 0.9728 | F1: 0.9421 | AUC: 0.8433 Epoch 18/20 Train Loss: 0.0337 | Train Acc: 0.9873 Val Loss: 0.5600 | Val Acc: 0.8895 Precision: 0.9280 | Recall: 0.9456 | F1: 0.9367 | AUC: 0.8400 Epoch 19/20 Train Loss: 0.0333 | Train Acc: 0.9875 Val Loss: 0.6489 | Val Acc: 0.9007 Precision: 0.9165 | Recall: 0.9740 | F1: 0.9443 | AUC: 0.8449 Epoch 20/20 Train Loss: 0.0093 | Train Acc: 0.9967 Val Loss: 0.7390 | Val Acc: 0.8987 Precision: 0.9108 | Recall: 0.9787 | F1: 0.9435 | AUC: 0.8396  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   74 / 77

--- End of Page 74 ---

In [75]:   history_df = pd.DataFrame(history) history_df.to_csv(      "/kaggle/working/training_history.csv",      index=False ) print(history_df)       epoch  train_loss  train_accuracy  val_loss  val_accuracy    precision  \ 0       1    0.391946        0.863786  0.393708      0.864893     0.864893    1       2    0.331657        0.875411  0.329729      0.875128     0.885806    2       3    0.294872        0.890546  0.307874      0.886387     0.888771    3       4    0.250750        0.904365  0.298769      0.882293     0.912896    4       5    0.234646        0.912919  0.298035      0.897646     0.900107    5       6    0.199770        0.928054  0.277565      0.903787     0.918618    6       7    0.179183        0.937925  0.306616      0.895599     0.913237    7       8    0.149231        0.948015  0.384353      0.893552     0.893730    8       9    0.149451        0.946699  0.362153      0.894575     0.898069    9      10    0.140277        0.950208  0.304227      0.905834     0.915105    10     11    0.101851        0.962492  0.383680      0.878199     0.936298    11     12    0.106852        0.960298  0.306127      0.903787     0.935110    12     13    0.078529        0.973459  0.348935      0.893552     0.922463    13     14    0.096419        0.964466  0.610851      0.891505     0.892667    14     15    0.064340        0.978504  0.493851      0.899693     0.906420    15      6    0.065274        0.977846  0.460238      0.899693     0.909989    16      7    0.064604        0.978065  0.477429      0.900716     0.919283    17      8    0.052946        0.981136  0.543183      0.899693     0.902913    18      9    0.038157        0.986181  0.508250      0.899693     0.923950    19     10    0.057236        0.979601  0.580644      0.900716     0.909190    20     11    0.029728        0.989691  0.594198      0.905834     0.917869    21     12    0.031576        0.988813  0.547739      0.903787     0.926220    22     13    0.043664        0.983549  0.600371      0.896622      8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   75 / 77

--- End of Page 75 ---

0.914254    23     14    0.028372        0.989910  0.605758      0.891505     0.915636    24     15    0.022164        0.991226  0.645465      0.893552     0.915825    25     16    0.029279        0.990787  0.703693      0.898669     0.906318    26     17    0.029336        0.988813  0.485614      0.896622     0.913333    27     18    0.033661        0.987278  0.559978      0.889458     0.927991    28     19    0.033344        0.987497  0.648909      0.900716     0.916481    29     20    0.009313        0.996710  0.738953      0.898669     0.910793           recall        f1       auc   0   1.000000  0.927552  0.725108   1   0.982249  0.931538  0.788462   2   0.992899  0.937954  0.818800   3   0.955030  0.933488  0.832948   4   0.991716  0.943694  0.832706   5   0.975148  0.946039  0.849561   6   0.971598  0.941514  0.850520   7   0.995266  0.941769  0.846696   8   0.990533  0.942037  0.844737   9   0.982249  0.947489  0.848557   10  0.921893  0.929040  0.855074   11  0.955030  0.944965  0.857531   12  0.957396  0.939605  0.844513   13  0.994083  0.940649  0.820042   14  0.985799  0.944444  0.837556   15  0.981065  0.944191  0.846315   16  0.970414  0.944157  0.841330   17  0.990533  0.944695  0.830312   18  0.963314  0.943221  0.837708   19  0.983432  0.944855  0.839345   20  0.978698  0.947308  0.842855   21  0.965680  0.945539  0.851385   22  0.971598  0.942054  0.837861   23  0.963314  0.938870  0.840779   24  0.965680  0.940092  0.838054   25  0.984615  0.943846  0.826533   26  0.972781  0.942120  0.843339   27  0.945562  0.936694  0.840004   28  0.973964  0.944349  0.844939   29  0.978698  0.943525  0.839636    8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   76 / 77

--- End of Page 76 ---

Exported with  runcell  — convert notebooks to HTML or PDF anytime at runcell.dev.  In [76]:   best_row = history_df.loc[      history_df["f1"].idxmax() ] print("Best Epoch:", int(best_row["epoch"])) print("Accuracy:", best_row["val_accuracy"]) print("Precision:", best_row["precision"]) print("Recall:", best_row["recall"]) print("F1:", best_row["f1"]) print("AUC:", best_row["auc"])  Best Epoch: 10 Accuracy: 0.9058341862845445 Precision: 0.9151047409040793 Recall: 0.9822485207100592 F1: 0.9474885844748858 AUC: 0.8485565716334948  8/15/26, 11:49 AM   Jupyter Notebook — generated with runcell  about:blank   77 / 77